<a href="https://colab.research.google.com/github/davidfague/Neural-Modeling/blob/load_synapses/notebooks/AA_pre_sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook will be used to build your simulation.

In [1]:
# if running in colab:
!git clone https://github.com/davidfague/Neural-Modeling.git -b load_synapses
# !pip install -r Neural-Modeling/setup/requirements.txt #-c Neural-Modeling/setup/Colab_install_constraints.txt
# !grep -v -i -E '^(numpy|matplotlib|psutil|h5py|scikit-learn|seaborn|tqdm|netpyne)' Neural-Modeling/setup/requirements.txt | pip install --no-deps -r /dev/stdin
%cd Neural-Modeling/notebooks

Cloning into 'Neural-Modeling'...
remote: Enumerating objects: 5969, done.
remote: Counting objects: 100% (385/385), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 5969 (delta 336), reused 280 (delta 268), pack-reused 5584 (from 2)
Receiving objects: 100% (5969/5969), 88.27 MiB | 11.30 MiB/s, done.
Resolving deltas: 100% (4015/4015), done.
/content/Neural-Modeling/notebooks


In [2]:
!pip install neuron

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 63.4 MB/s eta 0:00:00


In [3]:
!pip install neuron_reduce

In [4]:
import sys
sys.path.append('..')
sys.path.append('../Modules')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print(os.getcwd())
if 'notebook' in os.getcwd():
    # os.chdir("../scripts") # go from Neural-Modeling/notebooks to Neural-Modeling/scripts, where simulation outputs will be generated to. (maybe a separate folder could be used...)
    os.chdir("../simulations") # go to output folder
    print(os.getcwd())

/content/Neural-Modeling/notebooks
/content/Neural-Modeling/simulations


## User Specifications -  parameters, simulation folder

In [18]:
from Modules.constants import HayParameters
import datetime
import pickle
from neuron import h

from Modules.simulation_slurm import Simulator

from scripts.gen_param_list_advanced import sim_type_params_all

# this is a quick example, but the advanced generation of parameter sets and titles for parametric study is in scripts/gen_param_list_advanced.py
sim_set_title= "baseline_no_clusters"# "name_of_simulation_set"
sim_titles =  ["complex"] #["name_of_simulation1_within_set"] # for multiple: ["description_of_simulation1_within_set", "description_of_simulation2_within_set"],
sim_type = 'sta' # select simulation type

# import predefined clusters. stored in Modules/clusters.py instead of default from Modules/constants.py.
from Modules.clusters import exc_clustering#, inh_clustering

parameter_sets = [HayParameters(sim_title,
                                all_synapses_off=True,
                                exc_clustering=exc_clustering,  # just use one big exc cluster for now.
                                # inh_clustering=inh_clustering,
                                **sim_type_params_all[sim_type],
                                )
                                for sim_title in sim_titles] # no variation between simulations with this code snippet
# initialize with all_synapses_off for saving segments.csv without building synapses. # TODO: adjust saving of segments.csv to not require this. Also requires removing previous built-in synapse building implementation from cell_builder.py.

simulator = Simulator(
                        sim_set_title = sim_set_title,
                        sim_titles = sim_titles,
                        parameter_sets = parameter_sets)

# create simulation folders and save parameters in them
simulator.create_simulation_folders()

# @TODO: modularize gen_param_list_advanced.py so it can optionally be used here.
# @TODO: compile modfiles once in Simulator.__init__ if not already compiled. would need to specify them.
# @TODO: implement running accompanying functions on all sims within the simulator object in parallel or loop.

ImportError: cannot import name 'contains_only_chunked_or_numpy' from 'xarray.core.utils' (/usr/local/lib/python3.11/dist-packages/xarray/core/utils.py)

In [ ]:
sim_type_params_all[sim_type]

In [ ]:
simulator.sims_dir

In [ ]:
sim_dir = os.path.join(simulator.sims_dir, simulator.sim_titles[0]) # choose first simulation directory for the example.
os.path.abspath(sim_dir) # for copying over

## Generate segments.csv

In [ ]:
from Modules.segments_file import generate_segments_csv

simulator.run_on_all_sims(simulator.sims_dir, generate_segments_csv) # generate the segments csv for each simulation

In [ ]:
# read segments.csv file
seg_data = pd.read_csv(os.path.join(sim_dir, "segment_data.csv"))
seg_data.head() # show the first few rows of the segments data

In [ ]:
from Modules import analysis
parameters = analysis.DataReader.load_parameters(sim_dir) # load parameters
# parameters # print parameters

## Visualize Cell and automated classification of dendritic types

In [ ]:
# # for visualizing the cell and automated sec_type_precise generation
from Modules.plot_morphology import plot_morphology_with_highlighted_sec_type, plot_morphology_with_y_range
import Modules.analysis as analysis

# In the event that automation is not successful, we can deal with overlapping precise section types using the `overlaps` dictionary from the segments file.
# ### OVERLAPPING PRECISE SEC TYPE LABELS TODO: (pull overlaps from generate_segments_csv if there are overlaps)
# # overlapping_seg_ids = [i for i,j in overlaps.items()]

################################
# highlight multiple section types (Figure for AST)
################################

# # Option 1: highlight multiple section types with different colors
# from Modules.plot_morphology import plot_morphology_with_highlighted_sec_types
# sec_types = ['perisomatic', 'distal_basal', 'distal_apical']
# fig, ax = plot_morphology_with_highlighted_sec_types(sec_types, seg_data, figsize=(10, 6))

# # Option 2: highlight multiple section types with the same color
# os.makedirs(os.path.join(sim_dir, "morphology"), exist_ok=True)
# for sec_type in parameters.inh_syn_properties.keys(): # get list of section types
#     print(f"Plotting {sec_type}:")
#     fig, ax = plot_morphology_with_highlighted_sec_type(sec_type, seg_data)
#     ax.set_title(sec_type) # TODO: not working for some reason
#     fig.tight_layout()
#     plt.show()
#     fig.savefig(os.path.join(sim_dir, "morphology", f"{sec_type}.png"))

################################
# highlight y_range in morphology (high Ca2+)
################################
# # Custom y-range (default is Ca2+ hot spot from hay et al. 2011)
# fig, ax = plot_morphology_with_y_range(seg_data, y_min=685, y_max=885) # can save to file with argument save_path='path/to/save.png'

################################
######### plot nexus ###########
################################
# sec_type = 'nexus'
# fig, ax = plot_morphology_with_highlighted_sec_type(sec_type, seg_data)
# ax.set_title(sec_type) # TODO: not working for some reason
# fig.tight_layout()
# plt.show()
# # fig.savefig(os.path.join(sim_dir, "morphology", f"{sec_type}.png"))

## Generate Synapses.csv

In [ ]:
# generate synapse objects abstractly. store info in csv.
from Modules.synapses_file import PreSimSynapseGenerator

# create PreSimSynapseGenerator instance and generate synapse locations, weights, etc.
# everything except presynaptic spike trains.
pssg = PreSimSynapseGenerator(sim_dir)
pssg.generate_synapse_locations()
pssg.synapses.to_csv(os.path.join(sim_dir, "synapses.csv"), index=False)

In [ ]:
# # view synapses from synapses csv
synapses = pd.read_csv(os.path.join(sim_dir, "synapses.csv"))
synapses.head() # show the first few lines of synapses

# tooltip: P_0 is release probability, initW is the initial weight multiplier,
# gbar_ampa and gbar_nmda are the excitatory synapse conductances,
# gbar_gaba is the inhibitory synapse conductance
# seg_id is the segment ID where the synapse is located

## Add spike trains to synapses.csv

In [ ]:
# adds column spike_train according to parameters.
pssg.generate_spike_trains_for_synapses()
# simulator.run_on_all_sims(simulator.sims_dir, use_pssg, pssg_function_name= 'generate_spike_trains_for_synapses') #TODO: fix

In [ ]:
# view updated synapses now having spike_train column
synapses = pd.read_csv(os.path.join(sim_dir, "synapses.csv"))
synapses.head() # show the csv

## Analyze Designed Synapses

### Visualize Cluster locations

In [ ]:
# parameters.exc_clustering # view the setup

In [ ]:
# parameters.inh_clustering # view the setup

In [ ]:
# TODO: move to synapse_analysis.py
# Import necessary modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from Modules.plot_morphology import plot_clusters
import os
import pickle

# Load parameters and data
# sim_dir = "path_to_your_simulation_directory"  # Replace with your actual simulation directory
with open(os.path.join(sim_dir, "parameters.pickle"), 'rb') as file:
    parameters = pickle.load(file)

# Load segment data
seg_data = pd.read_csv(os.path.join(sim_dir, "segment_data.csv"))

# # Load synapse data if available
synapses_with_seg_info = synapses.merge( # TODO: alternative could be used to save memory.
    seg_data,
    on='seg_id',
    how='left',               # carry along all synapses even if a seg_id is missing
    suffixes=('','_seg')      # if both synapses and segments have a column with the same name, suffix the seg col with _seg
)

columns = ['pc_0', 'pc_1', 'pc_2']

# get the coordinates of the synapses
synapse_coords = synapses_with_seg_info[columns].values

# Create figure with multiple subplots for different section types
fig = plt.figure(figsize=(20, 10))

# Plot excitatory clusters
ax1 = fig.add_subplot(121, projection='3d')
plot_clusters(
    seg_data=seg_data,
    clustering_config=parameters.exc_clustering,
    synapse_coords=synapse_coords,
    ax=ax1,
    elevation=20,
    azimuth=-100,
    title='Excitatory Clusters'
)

# Plot inhibitory clusters
ax2 = fig.add_subplot(122, projection='3d')
plot_clusters(
    seg_data=seg_data,
    clustering_config=parameters.inh_clustering,
    synapse_coords=synapse_coords,
    ax=ax2,
    elevation=20,
    azimuth=-100,
    title='Inhibitory Clusters'
)

plt.tight_layout()
plt.show()

# # Optional: Plot individual section types separately
# # exc clusters
# for sec_type in parameters.exc_clustering.keys(): #exc clusters
#     fig = plt.figure(figsize=(10, 10))
#     ax = fig.add_subplot(111, projection='3d')
#     plot_clusters(
#         seg_data=seg_data,
#         clustering_config={sec_type: parameters.exc_clustering[sec_type]},
#         synapse_coords=synapse_coords,
#         ax=ax,
#         elevation=20,
#         azimuth=-100,
#         title=f'Excitatory Clusters - {sec_type}'
#     )
#     plt.tight_layout()
#     plt.show()

# # inh clusters
# for sec_type in parameters.inh_clustering.keys(): # inh clusters
#     fig = plt.figure(figsize=(10, 10))
#     ax = fig.add_subplot(111, projection='3d')
#     plot_clusters(
#         seg_data=seg_data,
#         clustering_config={sec_type: parameters.inh_clustering[sec_type]},
#         synapse_coords=synapse_coords,
#         ax=ax,
#         elevation=20,
#         azimuth=-100,
#         title=f'Inhibitory Clusters - {sec_type}'
#     )
#     plt.tight_layout()
#     plt.show()

## Further synapse design analysis (Spike raster, etc.) (To be finished)

In [ ]:
from Modules.synapse_analysis import SynapseAnalyzer

# Initialize the analyzer with your simulation directory
analyzer = SynapseAnalyzer(sim_dir)
analyzer.add_segment_data()

In [ ]:
# Get statistics for a specific functional group
stats = analyzer.analyze_cluster_statistics(
    functional_group_id=0,
    synapse_type='exc'
)
print(f"stats for exc cluster group {0}: {stats}")

In [ ]:
# Generate a spike raster plot for excitatory synapses
analyzer.plot_spike_raster(
    synapse_types=['exc'],
    time_window=(0, 1000),  # First second of simulation
    save_path=os.path.join(sim_dir, 'spike_raster.png')
)

In [ ]:
# Plot the spatial distribution of synapses belonging to clusters
analyzer.plot_cluster_spatial_distribution(
    synapse_type='exc',
    save_path=os.path.join(sim_dir, 'spatial_distribution.png')
)

In [ ]:
# Analyze correlations between spike trains
analyzer.plot_correlation_matrix(
    synapse_type='exc',
    time_window=(0, 1000),
    save_path=os.path.join(sim_dir, 'correlation_matrix.png')
)

## Copy over the sim_dir to notebooks/AA_sim.ipynb then notebooks/AA_post_sim.ipynb for simulation and analysis.

In [ ]:
os.path.abspath(sim_dir) # for copying over